In [1]:
# ============================================================
# 🚌 SMART VOICE BUS ASSISTANT — ALL TAMIL NADU DISTRICTS
# STT + NLU + TTS + Smart Route Reasoning
# Whisper + gTTS + Real Dataset + GenAI Logic
# ============================================================

# ============================================================
# STEP 1 — INSTALL LIBRARIES
# ============================================================

!pip install openai-whisper gTTS SpeechRecognition pandas requests sounddevice scipy ipywidgets fuzzywuzzy python-Levenshtein -q

!apt-get install -y portaudio19-dev -q
!pip install pyaudio -q

print("✅ Libraries Installed")


# ============================================================
# STEP 2 — IMPORTS
# ============================================================

import pandas as pd
import requests
import random
import os
import re
from datetime import datetime
from fuzzywuzzy import process

print("✅ Imports Done")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 8.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 27.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.2 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.
Reading package lists...
Building dependency tree...
R

In [4]:
# ============================================================
# STEP 3 — REAL + HYBRID DATASET PIPELINE
# Using uploaded SETC dataset + synthetic town bus data
# ============================================================

import pandas as pd
import random

# ============================================================
# LOAD REAL SETC DATASET
# ============================================================

real_df = pd.read_csv("/content/SETCbustimings_1_0.csv")

print("✅ REAL SETC DATASET LOADED")
print(real_df.head())

# ============================================================
# CLEAN & RENAME COLUMNS
# ============================================================

real_df.columns = [
    c.strip().lower().replace(" ", "_")
    for c in real_df.columns
]

rename_map = {

    "route_no.": "bus_no",
    "route_no": "bus_no",
    "from": "from_stop",
    "to": "to_stop",
    "departure_timings": "departure",
    "type": "type"

}

real_df.rename(columns=rename_map, inplace=True)

# ============================================================
# KEEP ONLY REQUIRED COLUMNS
# ============================================================

required_cols = [
    "bus_no",
    "from_stop",
    "to_stop",
    "departure",
    "type"
]

real_df = real_df[required_cols]

# ============================================================
# CLEAN VALUES
# ============================================================

for col in ["bus_no", "from_stop", "to_stop", "departure", "type"]:
    real_df[col] = real_df[col].astype(str).str.strip()

print("\n✅ CLEANED REAL DATASET")
print(real_df.head())

# ============================================================
# SYNTHETIC TOWN BUS DATA
# ============================================================

districts = [

    "Chennai",
    "Coimbatore",
    "Madurai",
    "Salem",
    "Erode",
    "Tiruppur",
    "Trichy",
    "Karur",
    "Namakkal",
    "Thoothukudi",
    "Tirunelveli",
    "Kanyakumari",
    "Dindigul",
    "Thanjavur",
    "Vellore",
    "Villupuram",
    "Cuddalore",
    "Krishnagiri",
    "Dharmapuri",
    "Ooty",
    "Pollachi",
    "Hosur",
    "Nagercoil"

]

town_bus_types = [
    "Town Bus",
    "Mini Bus",
    "Ordinary"
]

synthetic_rows = []

bus_counter = 5000

for source in districts:

    nearby_places = random.sample(districts, 5)

    for destination in nearby_places:

        if source != destination:

            for trip in range(5):

                dep_hour = random.randint(5, 22)
                dep_min = random.randint(0, 59)

                synthetic_rows.append({

                    "bus_no":
                    f"TB{bus_counter}",

                    "from_stop":
                    source,

                    "to_stop":
                    destination,

                    "departure":
                    f"{dep_hour:02d}:{dep_min:02d}",

                    "type":
                    random.choice(town_bus_types)

                })

                bus_counter += 1

# ============================================================
# CREATE SYNTHETIC DATAFRAME
# ============================================================

synthetic_df = pd.DataFrame(synthetic_rows)

print("\n✅ SYNTHETIC TOWN BUS DATA CREATED")
print(synthetic_df.head())

# ============================================================
# COMBINE BOTH DATASETS
# ============================================================

df_bus = pd.concat(
    [real_df, synthetic_df],
    ignore_index=True
)

# ============================================================
# REMOVE DUPLICATES
# ============================================================

df_bus.drop_duplicates(inplace=True)

# ============================================================
# SAVE FINAL DATASET
# ============================================================

df_bus.to_csv(
    "final_tamilnadu_bus_dataset.csv",
    index=False
)

# ============================================================
# FINAL OUTPUT
# ============================================================

print("\n================================================")
print("✅ FINAL HYBRID DATASET READY")
print("================================================")

print("\n📊 FINAL DATASET SHAPE:")
print(df_bus.shape)

print("\n🚌 SAMPLE RECORDS:")
print(df_bus.sample(10))

print("\n✅ BUS TYPES AVAILABLE:")
print(df_bus["type"].unique())

print("\n✅ TOTAL UNIQUE ROUTES:")
print(
    len(
        df_bus[
            ["from_stop", "to_stop"]
        ].drop_duplicates()
    )
)

print("\n✅ DATASET SAVED:")
print("final_tamilnadu_bus_dataset.csv")

✅ REAL SETC DATASET LOADED
   Sl. No. Depot Route No.        From          To  Route Length   Type  \
0        1   SHN     192UD   ALANKULAM     CHENNAI           657  ULTRA   
1        2   TRY     127UD   ANNAVASAL     CHENNAI           384  ULTRA   
2        3   CBE     479UD       ARANI  COIMBATORE           394  ULTRA   
3        4    CB     131UD  ARANTHANGI     CHENNAI           417  ULTRA   
4        5  KKDI     126AU    ARIMALAM     CHENNAI           403  ULTRA   

   No.of Service Departure Timings  
0              1             17.45  
1              1              21.3  
2              1                21  
3              1              20.3  
4              1             19.15  

✅ CLEANED REAL DATASET
  bus_no   from_stop     to_stop departure   type
0  192UD   ALANKULAM     CHENNAI     17.45  ULTRA
1  127UD   ANNAVASAL     CHENNAI      21.3  ULTRA
2  479UD       ARANI  COIMBATORE        21  ULTRA
3  131UD  ARANTHANGI     CHENNAI      20.3  ULTRA
4  126AU    ARIMALAM     C

In [5]:
# ============================================================
# STEP 4 — NLU + INTENT DETECTION
# ============================================================
# This step:
# ✅ Understands Tamil + English queries
# ✅ Detects:
#    - Route query
#    - Bus timing query
#    - Next arrival query
# ✅ Extracts:
#    - Bus number
#    - From stop
#    - To stop
# ============================================================

import re

# ============================================================
# STOP NAME ALIASES
# ============================================================

STOP_ALIASES = {

    "chennai":
    ["chennai", "madras", "சென்னை"],

    "coimbatore":
    ["coimbatore", "kovai", "கோவை"],

    "madurai":
    ["madurai", "மதுரை"],

    "trichy":
    ["trichy", "tiruchy", "திருச்சி"],

    "salem":
    ["salem", "சேலம்"],

    "erode":
    ["erode", "ஈரோடு"],

    "tiruppur":
    ["tiruppur", "திருப்பூர்"],

    "karur":
    ["karur", "காரூர்"],

    "namakkal":
    ["namakkal", "நாமக்கல்"],

    "vellore":
    ["vellore", "வேலூர்"],

    "ooty":
    ["ooty", "உட்டி", "ஊட்டி"],

    "pollachi":
    ["pollachi", "பொள்ளாச்சி"],

    "hosur":
    ["hosur", "ஓசூர்"],

    "nagercoil":
    ["nagercoil", "நாகர்கோவில்"]

}

# ============================================================
# NORMALISE STOP NAME
# ============================================================

def normalize_stop(text):

    text = text.lower()

    for canonical, variants in STOP_ALIASES.items():

        for variant in variants:

            if variant in text:

                return canonical.capitalize()

    return None

# ============================================================
# EXTRACT BUS NUMBER
# ============================================================

def extract_bus_number(text):

    match = re.search(
        r'\b([A-Za-z]{0,3}[0-9]{1,5}[A-Za-z]{0,3})\b',
        text
    )

    if match:
        return match.group(1).upper()

    return None

# ============================================================
# DETECT INTENT
# ============================================================

def detect_intent(text):

    text = text.lower()

    # --------------------------------------------------------
    # NEXT ARRIVAL
    # --------------------------------------------------------

    next_keywords = [

        "next bus",
        "when comes",
        "when will",
        "eppo varum",
        "arrival",
        "next arrival",
        "அடுத்த",
        "வரும்"

    ]

    # --------------------------------------------------------
    # TIMING QUERY
    # --------------------------------------------------------

    timing_keywords = [

        "timing",
        "time",
        "schedule",
        "departure",
        "eppo",
        "நேரம்",
        "எப்போ"

    ]

    # --------------------------------------------------------
    # ROUTE QUERY
    # --------------------------------------------------------

    route_keywords = [

        "which bus",
        "route",
        "goes to",
        "bus from",
        "எந்த பஸ்"

    ]

    # --------------------------------------------------------
    # CHECK
    # --------------------------------------------------------

    if any(word in text for word in next_keywords):
        return "next_arrival"

    if any(word in text for word in timing_keywords):
        return "bus_timing"

    if any(word in text for word in route_keywords):
        return "route_query"

    # If bus number exists → timing query

    if extract_bus_number(text):
        return "bus_timing"

    # fallback

    return "route_query"

# ============================================================
# PARSE QUERY
# ============================================================

def parse_query(text):

    intent = detect_intent(text)

    bus_no = extract_bus_number(text)

    # --------------------------------------------------------
    # FIND LOCATIONS
    # --------------------------------------------------------

    found_stops = []

    for canonical, variants in STOP_ALIASES.items():

        for variant in variants:

            if variant.lower() in text.lower():

                found_stops.append(
                    canonical.capitalize()
                )

                break

    found_stops = list(dict.fromkeys(found_stops))

    from_stop = None
    to_stop = None

    if len(found_stops) >= 1:
        from_stop = found_stops[0]

    if len(found_stops) >= 2:
        to_stop = found_stops[1]

    # --------------------------------------------------------
    # RETURN
    # --------------------------------------------------------

    return {

        "intent": intent,

        "bus_no": bus_no,

        "from_stop": from_stop,

        "to_stop": to_stop

    }

# ============================================================
# TEST
# ============================================================

test_queries = [

    "Chennai to Coimbatore bus",

    "192UD bus timing",

    "Next bus from Madurai to Chennai",

    "திருப்பூர் to கோவை எந்த பஸ்?",

    "TNSTC234 eppo varum?"

]

print("================================================")
print("✅ NLU TEST")
print("================================================\n")

for q in test_queries:

    result = parse_query(q)

    print("QUERY:")
    print(q)

    print("\nPARSED RESULT:")
    print(result)

    print("\n--------------------------------------------\n")

✅ NLU TEST

QUERY:
Chennai to Coimbatore bus

PARSED RESULT:
{'intent': 'route_query', 'bus_no': None, 'from_stop': 'Chennai', 'to_stop': 'Coimbatore'}

--------------------------------------------

QUERY:
192UD bus timing

PARSED RESULT:
{'intent': 'bus_timing', 'bus_no': '192UD', 'from_stop': None, 'to_stop': None}

--------------------------------------------

QUERY:
Next bus from Madurai to Chennai

PARSED RESULT:
{'intent': 'next_arrival', 'bus_no': None, 'from_stop': 'Chennai', 'to_stop': 'Madurai'}

--------------------------------------------

QUERY:
திருப்பூர் to கோவை எந்த பஸ்?

PARSED RESULT:
{'intent': 'route_query', 'bus_no': None, 'from_stop': 'Coimbatore', 'to_stop': 'Tiruppur'}

--------------------------------------------

QUERY:
TNSTC234 eppo varum?

PARSED RESULT:
{'intent': 'next_arrival', 'bus_no': None, 'from_stop': None, 'to_stop': None}

--------------------------------------------



In [6]:
# ============================================================
# STEP 5 — BUS QUERY ENGINE
# ============================================================
# This step:
# ✅ Searches buses
# ✅ Finds route buses
# ✅ Finds next arrival
# ✅ Finds timings
# ✅ Handles unknown buses
# ✅ Supports Tamil + English queries
# ============================================================

from datetime import datetime

# ============================================================
# CONVERT TIME → MINUTES
# ============================================================

def time_to_minutes(time_str):

    try:

        # Handle 17.45 format
        time_str = str(time_str).replace(".", ":")

        h, m = map(int, time_str.split(":"))

        return h * 60 + m

    except:

        return 9999

# ============================================================
# MAIN QUERY ENGINE
# ============================================================

def query_bus_system(parsed):

    intent = parsed["intent"]

    bus_no = parsed["bus_no"]

    from_stop = parsed["from_stop"]

    to_stop = parsed["to_stop"]

    df = df_bus.copy()

    # --------------------------------------------------------
    # CLEAN
    # --------------------------------------------------------

    for col in ["bus_no", "from_stop", "to_stop"]:

        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
        )

    # ========================================================
    # ROUTE QUERY
    # ========================================================

    if intent == "route_query":

        results = df.copy()

        if from_stop:

            results = results[
                results["from_stop"]
                .str.lower()
                ==
                from_stop.lower()
            ]

        if to_stop:

            results = results[
                results["to_stop"]
                .str.lower()
                ==
                to_stop.lower()
            ]

        # remove duplicate buses

        results = results.drop_duplicates(
            subset=["bus_no"]
        )

        # ----------------------------------------------------
        # NO RESULT
        # ----------------------------------------------------

        if len(results) == 0:

            return (
                f"Sorry. No buses found "
                f"from {from_stop} to {to_stop}."
            )

        # ----------------------------------------------------
        # RESULT
        # ----------------------------------------------------

        buses = results["bus_no"].tolist()[:10]

        return (
            f"Buses available from "
            f"{from_stop} to {to_stop}: "
            f"{', '.join(buses)}"
        )

    # ========================================================
    # BUS TIMING QUERY
    # ========================================================

    elif intent == "bus_timing":

        results = df.copy()

        # ----------------------------------------------------
        # SEARCH BY BUS NUMBER
        # ----------------------------------------------------

        if bus_no:

            results = results[
                results["bus_no"]
                .str.upper()
                ==
                bus_no.upper()
            ]

        # ----------------------------------------------------
        # OPTIONAL FROM FILTER
        # ----------------------------------------------------

        if from_stop:

            results = results[
                results["from_stop"]
                .str.lower()
                ==
                from_stop.lower()
            ]

        # ----------------------------------------------------
        # NO RESULT
        # ----------------------------------------------------

        if len(results) == 0:

            return (
                f"Bus {bus_no} not found "
                f"in database."
            )

        # ----------------------------------------------------
        # GET TIMINGS
        # ----------------------------------------------------

        timings = results["departure"].tolist()[:10]

        first = results.iloc[0]

        return (

            f"Bus {first['bus_no']} "
            f"from {first['from_stop']} "
            f"to {first['to_stop']} "
            f"departure timings are: "
            f"{', '.join(map(str, timings))}"

        )

    # ========================================================
    # NEXT ARRIVAL QUERY
    # ========================================================

    elif intent == "next_arrival":

        results = df.copy()

        # ----------------------------------------------------
        # FILTER
        # ----------------------------------------------------

        if from_stop:

            results = results[
                results["from_stop"]
                .str.lower()
                ==
                from_stop.lower()
            ]

        if to_stop:

            results = results[
                results["to_stop"]
                .str.lower()
                ==
                to_stop.lower()
            ]

        # ----------------------------------------------------
        # NO RESULT
        # ----------------------------------------------------

        if len(results) == 0:

            return (
                f"No upcoming buses found "
                f"from {from_stop} to {to_stop}."
            )

        # ----------------------------------------------------
        # CURRENT TIME
        # ----------------------------------------------------

        now = datetime.now()

        current_minutes = (
            now.hour * 60
            +
            now.minute
        )

        # ----------------------------------------------------
        # CONVERT DEPARTURE
        # ----------------------------------------------------

        results["dep_minutes"] = results[
            "departure"
        ].apply(time_to_minutes)

        # ----------------------------------------------------
        # FUTURE BUSES
        # ----------------------------------------------------

        future = results[
            results["dep_minutes"]
            >
            current_minutes
        ]

        # ----------------------------------------------------
        # IF NO FUTURE BUSES
        # ----------------------------------------------------

        if len(future) == 0:

            return (
                f"No more buses available today "
                f"from {from_stop} to {to_stop}."
            )

        # ----------------------------------------------------
        # FIND NEXT BUS
        # ----------------------------------------------------

        future = future.sort_values(
            by="dep_minutes"
        )

        next_bus = future.iloc[0]

        waiting_time = (
            next_bus["dep_minutes"]
            -
            current_minutes
        )

        return (

            f"Next bus is "
            f"{next_bus['bus_no']} "
            f"from {next_bus['from_stop']} "
            f"to {next_bus['to_stop']}. "
            f"It will arrive in "
            f"{waiting_time} minutes. "
            f"Departure time is "
            f"{next_bus['departure']}."

        )

    # ========================================================
    # FALLBACK
    # ========================================================

    return "Sorry. Query not understood."

# ============================================================
# TEST QUERIES
# ============================================================

test_queries = [

    "Chennai to Madurai bus",

    "192UD bus timing",

    "Next bus from Salem to Coimbatore",

    "TNSTC120 timing",

    "திருப்பூர் to கோவை எந்த பஸ்?"

]

print("================================================")
print("✅ BUS QUERY ENGINE TEST")
print("================================================\n")

for q in test_queries:

    print("QUERY:")
    print(q)

    parsed = parse_query(q)

    answer = query_bus_system(parsed)

    print("\nANSWER:")
    print(answer)

    print("\n--------------------------------------------\n")

✅ BUS QUERY ENGINE TEST

QUERY:
Chennai to Madurai bus

ANSWER:
Buses available from Chennai to Madurai: 137AC, 137UD

--------------------------------------------

QUERY:
192UD bus timing

ANSWER:
Bus 192UD from ALANKULAM to CHENNAI departure timings are: 17.45, 17.4

--------------------------------------------

QUERY:
Next bus from Salem to Coimbatore

ANSWER:
No upcoming buses found from Coimbatore to Salem.

--------------------------------------------

QUERY:
TNSTC120 timing

ANSWER:
Bus 192UD from ALANKULAM to CHENNAI departure timings are: 17.45, 21.3, 21, 20.3, 19.15, 19.00,20.00, 20, 19, 20.15, 07.30,08.00,10.30,22.30,23.00

--------------------------------------------

QUERY:
திருப்பூர் to கோவை எந்த பஸ்?

ANSWER:
Sorry. No buses found from Coimbatore to Tiruppur.

--------------------------------------------



In [7]:
# ============================================================
# STEP 6 — TEXT TO SPEECH (TTS)
# ============================================================
# This step:
# ✅ Converts AI response → Voice
# ✅ Tamil + English support
# ✅ Speaks bus timings
# ✅ Speaks next arrival
# ✅ Uses gTTS
# ============================================================

from gtts import gTTS
import IPython.display as ipd
import os

# ============================================================
# LANGUAGE DETECTION
# ============================================================

def detect_language(text):

    tamil_count = 0

    for char in text:

        if '\u0B80' <= char <= '\u0BFF':

            tamil_count += 1

    # --------------------------------------------------------
    # TAMIL
    # --------------------------------------------------------

    if tamil_count > 2:

        return "ta"

    # --------------------------------------------------------
    # ENGLISH
    # --------------------------------------------------------

    return "en"

# ============================================================
# TEXT TO SPEECH
# ============================================================

def speak_answer(answer_text):

    # --------------------------------------------------------
    # DETECT LANGUAGE
    # --------------------------------------------------------

    lang = detect_language(answer_text)

    print("================================================")
    print("🔊 SPEAKING RESPONSE")
    print("================================================")

    print("\nLANGUAGE:")
    print(lang)

    print("\nTEXT:")
    print(answer_text)

    # --------------------------------------------------------
    # CREATE TTS
    # --------------------------------------------------------

    tts = gTTS(

        text=answer_text,

        lang=lang,

        slow=False

    )

    # --------------------------------------------------------
    # SAVE AUDIO
    # --------------------------------------------------------

    output_file = "bus_response.mp3"

    tts.save(output_file)

    print("\n✅ AUDIO GENERATED")

    # --------------------------------------------------------
    # PLAY AUDIO
    # --------------------------------------------------------

    return ipd.Audio(
        output_file,
        autoplay=True
    )

# ============================================================
# TEST TTS
# ============================================================

sample_answers = [

    "Bus 192UD from Alangulam to Chennai departure timing is 17:45.",

    "Next bus from Chennai to Madurai will arrive in 20 minutes.",

    "திருப்பூரில் இருந்து கோவைக்கு செல்லும் அடுத்த பஸ் 15 நிமிடங்களில் வரும்."

]

print("================================================")
print("✅ TTS TEST")
print("================================================")

for ans in sample_answers:

    audio = speak_answer(ans)

    display(audio)

    print("\n--------------------------------------------\n")

✅ TTS TEST
🔊 SPEAKING RESPONSE

LANGUAGE:
en

TEXT:
Bus 192UD from Alangulam to Chennai departure timing is 17:45.

✅ AUDIO GENERATED



--------------------------------------------

🔊 SPEAKING RESPONSE

LANGUAGE:
en

TEXT:
Next bus from Chennai to Madurai will arrive in 20 minutes.

✅ AUDIO GENERATED



--------------------------------------------

🔊 SPEAKING RESPONSE

LANGUAGE:
ta

TEXT:
திருப்பூரில் இருந்து கோவைக்கு செல்லும் அடுத்த பஸ் 15 நிமிடங்களில் வரும்.

✅ AUDIO GENERATED



--------------------------------------------



In [13]:
# ============================================================
# STEP 7 — LIVE MICROPHONE STT (BETTER UI)
# ============================================================

!pip install openai-whisper -q

import whisper
from IPython.display import display, Javascript, HTML
from google.colab import output
from base64 import b64decode

# ============================================================
# LOAD WHISPER MODEL
# ============================================================

print("Loading Whisper model...")

model = whisper.load_model("base")

print("✅ Whisper model loaded")

# ============================================================
# HTML UI
# ============================================================

display(HTML("""

<h2 style="color:green;">🎤 Smart Voice Bus Assistant</h2>

<p><b>Instructions:</b></p>

<ul>
<li>Click START RECORDING</li>
<li>Allow microphone permission</li>
<li>Speak clearly</li>
<li>Recording duration = 8 seconds</li>
</ul>

"""))

# ============================================================
# JAVASCRIPT RECORDER
# ============================================================

RECORD = """

async function recordAudio(){

    const div = document.createElement('div');

    div.innerHTML = `
    <button id="startBtn"
    style="
    background:green;
    color:white;
    padding:12px;
    font-size:18px;
    border:none;
    border-radius:8px;">
    START RECORDING
    </button>

    <p id="status"></p>
    `;

    document.body.appendChild(div);

    const startBtn = document.getElementById("startBtn");

    const status = document.getElementById("status");

    return new Promise(async(resolve)=>{

        startBtn.onclick = async()=>{

            startBtn.disabled = true;

            const stream =
            await navigator.mediaDevices.getUserMedia({audio:true});

            const recorder =
            new MediaRecorder(stream);

            const chunks = [];

            recorder.ondataavailable = e => chunks.push(e.data);

            recorder.start();

            // Countdown
            for(let i=8;i>=1;i--){

                status.innerHTML =
                "🎙️ Recording... " + i + " sec remaining";

                await new Promise(r=>setTimeout(r,1000));
            }

            recorder.stop();

            recorder.onstop = async()=>{

                status.innerHTML = "✅ Recording completed";

                const blob = new Blob(chunks);

                const reader = new FileReader();

                reader.readAsDataURL(blob);

                reader.onloadend = ()=>{

                    resolve(reader.result);

                };
            };
        };
    });
}
"""

display(Javascript(RECORD))

# ============================================================
# START RECORDING
# ============================================================

audio_data = output.eval_js("recordAudio()")

# ============================================================
# SAVE AUDIO
# ============================================================

binary = b64decode(audio_data.split(',')[1])

with open("recorded_audio.wav", "wb") as f:

    f.write(binary)

print("\n✅ Audio saved")

# ============================================================
# IMPROVED WHISPER TRANSCRIPTION
# ============================================================

!pip install ffmpeg-python -q

# ============================================================
# LOAD BETTER MODEL
# ============================================================

import whisper

print("Loading better Whisper model...")

# small = better accuracy than base
model = whisper.load_model("small")

print("✅ Better Whisper model loaded")

# ============================================================
# TRANSCRIBE WITH FORCED LANGUAGE
# ============================================================

print("\nTranscribing audio...")

result = model.transcribe(

    "recorded_audio.wav",

    language="en",     # force English
    task="transcribe",

    fp16=False

)

text = result["text"]

# ============================================================
# OUTPUT
# ============================================================

print("\n================================================")
print("🎯 FINAL TRANSCRIPTION")
print("================================================")

print("\nUser Speech:")
print(text)

Loading Whisper model...
✅ Whisper model loaded


<IPython.core.display.Javascript object>


✅ Audio saved
Loading better Whisper model...


100%|███████████████████████████████████████| 461M/461M [00:07<00:00, 64.9MiB/s]


✅ Better Whisper model loaded

Transcribing audio...

🎯 FINAL TRANSCRIPTION

User Speech:
 Bus route from Coimbatore to Madurai


In [16]:
# ============================================================
# BUS QUERY ENGINE
# ============================================================

import pandas as pd
import re
from datetime import datetime

# ============================================================
# LOAD DATASET
# ============================================================

df = pd.read_csv("final_tamilnadu_bus_dataset.csv")

print("✅ Dataset Loaded")
print(df.head())

# ============================================================
# NORMALIZE COLUMN NAMES
# ============================================================

df.columns = [c.lower().strip() for c in df.columns]

# ============================================================
# EXTRACT BUS NUMBER
# ============================================================

def extract_bus_number(text):

    pattern = r'\b[A-Za-z0-9]{2,10}\b'

    matches = re.findall(pattern, text)

    for m in matches:

        if any(char.isdigit() for char in m):
            return m.upper()

    return None

# ============================================================
# EXTRACT CITIES
# ============================================================

def extract_locations(text):

    cities = list(set(df["from_stop"].dropna().tolist() +
                      df["to_stop"].dropna().tolist()))

    found = []

    for city in cities:

        if str(city).lower() in text.lower():

            found.append(city)

    return found

# ============================================================
# FIND BUS
# ============================================================

def find_bus_info(parsed_query):

    bus_no = parsed_query["bus_no"]
    from_stop = parsed_query["from_stop"]
    to_stop = parsed_query["to_stop"]
    intent = parsed_query["intent"]

    # ========================================================
    # BUS NUMBER QUERY (for bus_timing or bus_number only)
    # ========================================================

    if bus_no and intent != "route_query": # If bus_no is the primary query, and not just part of a route query

        result = df[
            df["bus_no"].astype(str).str.upper() == bus_no
        ]

        if len(result) > 0:

            row = result.iloc[0]

            return f"""
Bus {row['bus_no']} runs from
{row['from_stop']} to {row['to_stop']}.

Departure timing is
{row['departure']}.
"""

        else:

            return "Sorry. Bus number not found."

    # ========================================================
    # SOURCE TO DESTINATION QUERY
    # ========================================================

    if from_stop and to_stop:

        result = df[
            (df["from_stop"].str.lower() == from_stop.lower()) &
            (df["to_stop"].str.lower() == to_stop.lower())
        ]

        if len(result) > 0:

            answer = f"Buses from {from_stop} to {to_stop} are:\n\n"

            for _, row in result.head(5).iterrows():

                answer += (
                    f"Bus {row['bus_no']} "
                    f"at {row['departure']}\n"
                )

            return answer

        else:

            return f"No buses found from {from_stop} to {to_stop}"

    # ========================================================
    # UNKNOWN QUERY
    # ========================================================

    return "Sorry, I could not understand your query."

# ============================================================
# TEST
# ============================================================

user_text_query = text # `text` comes from the STT output

print("\n================================================")
print("🎯 USER QUERY")
print("================================================")

print(user_text_query)

parsed_result = parse_query(user_text_query) # Use parse_query from NLU step
answer = find_bus_info(parsed_result)

print("\n================================================")
print("🚌 BUS ASSISTANT ANSWER")
print("================================================")

print(answer)

✅ Dataset Loaded
  bus_no   from_stop     to_stop departure   type
0  192UD   ALANKULAM     CHENNAI     17.45  ULTRA
1  127UD   ANNAVASAL     CHENNAI      21.3  ULTRA
2  479UD       ARANI  COIMBATORE        21  ULTRA
3  131UD  ARANTHANGI     CHENNAI      20.3  ULTRA
4  126AU    ARIMALAM     CHENNAI     19.15  ULTRA

🎯 USER QUERY
 Bus route from Coimbatore to Madurai

🚌 BUS ASSISTANT ANSWER
Buses from Coimbatore to Madurai are:

Bus TB5045 at 19:57
Bus TB5046 at 18:22
Bus TB5047 at 11:01
Bus TB5048 at 08:18
Bus TB5049 at 19:15



In [17]:
# ============================================================
# TEXT TO SPEECH
# ============================================================

from gtts import gTTS
from IPython.display import Audio

tts = gTTS(answer)

tts.save("response.mp3")

print("✅ Audio response generated")

Audio("response.mp3")

✅ Audio response generated
